<a href="https://colab.research.google.com/github/kader-xai/ml-course-notebooks/blob/main/nvidia_genai_course.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generative AI & LLMs — Code & Figures
### The reproducible programs behind the NCP-GENL course

This 48-episode course builds generative AI from first principles to a production RAG assistant on the NVIDIA stack. Most episodes are architecture diagrams, but **seven carry real, runnable code or a real figure** — and here they are, exactly as used in the videos:

| # | Episode | What it shows |
|---|---|---|
| NV03 | Tokenization | byte-level BPE on a sentence (tiktoken) |
| NV04 | Sampling | temperature & top-p on the same logits (figure) |
| NV05 | Scaling laws | the loss frontier + the emergence 'mirage' (figure) |
| NV10 | Attention | scaled dot-product attention from scratch (PyTorch) |
| NV16 | config.json | fingerprint real model configs (MHA/GQA/MoE/RoPE) |
| NV22 | QLoRA | why a 33B model fits on one 24GB card (memory math) |
| NV29 | Chunking | recursive splitter + cosine ranking (RAG ingestion) |

Run the setup cell once, then each episode runs top-to-bottom. A few cells are marked **[illustrative]** where the real path is GPU- or API-bound (a quantized 33B fine-tune, NIM embeddings) — the structure and numbers match the lesson; only the heavy compute is stood in for.

*— @kader_mohideen*

### Setup
Install the libraries the seven episodes use. (`torch` is already present in Colab.)

In [1]:
!pip install -q tiktoken numpy matplotlib transformers langchain-text-splitters

## NV03 — Tokenization & Vocabulary

A model never sees text — it sees token ids. Here byte-level BPE (tiktoken: GPT-4o's o200k vs GPT-2) splits a sentence into subword pieces, and you can see why a rarer language costs more tokens.

In [ ]:
import tiktoken
enc  = tiktoken.get_encoding("o200k_base")        # GPT-4o, ~200k byte-level BPE
gpt2 = tiktoken.get_encoding("gpt2")              # ~50k

text = "Tokenization splits text into subword units."
pieces = [enc.decode([t]) for t in enc.encode(text)]
print(pieces)
print(len(enc.encode(text)), "tokens for", len(text.split()), "words")

de = "Großbritannien verlässt die Europäische Union."
print("EN  o200k:", len(enc.encode(text)), "tokens")
print("DE  gpt2 :", len(gpt2.encode(de)), " o200k:", len(enc.encode(de)))

# chat models wrap text in role markers (Llama-3.1 format)
print("[BOS]user[EOH] Hi [EOT] assistant[EOH]")


## NV04 — Steering the Sampler — Temperature, Top-p, Top-k

The same logits, decoded three ways. Temperature reshapes the distribution (sharp → flat) and top-p (nucleus) truncates it. The figure shows all four side by side.

In [ ]:
import numpy as np, matplotlib
# matplotlib.use("Agg")  # inline in Colab
import matplotlib.pyplot as plt
np.random.seed(0)
toks  = ["Paris","the","a","France","located","situated","now","very"]
logit = np.log(np.array([0.55,0.14,0.10,0.08,0.055,0.035,0.025,0.015]))  # clean spread; T=1 -> Paris ~0.55
def sm(l,T): e=np.exp((l/ T)-(l/T).max()); return e/e.sum()
p03,p10,p15 = sm(logit,0.3), sm(logit,1.0), sm(logit,1.5)
print("T=0.3 Paris %.3f" % p03[0]); print("T=1.0 Paris %.3f the %.3f a %.3f France %.3f" % (p10[0],p10[1],p10[2],p10[3]))
print("T=1.5 Paris %.3f" % p15[0])
# top-p 0.9 nucleus on T=1.0
order=np.argsort(p10)[::-1]; cum=np.cumsum(p10[order]); k=int(np.searchsorted(cum,0.90)+1)
print("nucleus keeps", k, "of 8  (cum=%.3f)" % cum[k-1])

fig,ax=plt.subplots(2,2,figsize=(9,6.4)); fig.patch.set_facecolor("white")
BLUE="#2f6fdb"; GREY="#c7ccd6"
def bar(a,p,title,hi=True):
    a.bar(range(8), p, color=[BLUE if (hi and i==0) else "#7aa0e0" for i in range(8)], edgecolor="white")
    a.set_title(title, fontsize=12, fontweight="bold"); a.set_ylim(0,1.0)
    a.set_xticks(range(8)); a.set_xticklabels(toks, rotation=40, ha="right", fontsize=8); a.set_ylabel("probability", fontsize=9)
    a.spines[["top","right"]].set_visible(False)
bar(ax[0,0],p03,"T = 0.3  (sharp)"); ax[0,0].annotate("~0.98 on Paris", (0,p03[0]), xytext=(1.5,0.8), fontsize=9, color=BLUE)
bar(ax[0,1],p10,"T = 1.0  (natural)"); ax[0,1].annotate("~0.55", (0,p10[0]), xytext=(1.5,0.7), fontsize=9, color=BLUE)
bar(ax[1,0],p15,"T = 1.5  (flat)"); ax[1,0].annotate("~0.39, long-shots rise", (0,p15[0]), xytext=(1.2,0.7), fontsize=9, color=BLUE)
# panel 4: top-p nucleus on T=1.0 sorted
ps=p10[order]; cols=[BLUE if i<k else GREY for i in range(8)]
ax[1,1].bar(range(8), ps, color=cols, edgecolor="white"); ax[1,1].set_title("top-p = 0.9  (nucleus)", fontsize=12, fontweight="bold")
ax[1,1].set_xticks(range(8)); ax[1,1].set_xticklabels([toks[i] for i in order], rotation=40, ha="right", fontsize=8); ax[1,1].set_ylim(0,1.0)
ax[1,1].set_ylabel("probability", fontsize=9); ax[1,1].spines[["top","right"]].set_visible(False)
ax[1,1].annotate("kept %d of 8" % k, (k-0.5,0.4), fontsize=10, color=BLUE, fontweight="bold")
fig.suptitle("Decoding the SAME logits — temperature reshapes, top-p truncates", fontsize=12)
fig.text(0.99,0.01,"[illustrative — fixed seed]", ha="right", fontsize=8, color="#888")
plt.tight_layout(rect=[0,0.02,1,0.96])
fig.savefig("nv04.png", dpi=110, facecolor="white", bbox_inches="tight")
print("saved nvfig/nv04.png")
plt.show()


## NV05 — Scaling Laws & Emergence

The compute-optimal loss frontier is a straight line on a log-log plot; tokens/parameter reveals under- vs over-training; and 'emergence' looks sharp under a discontinuous metric but smooth under a continuous one — the emergence 'mirage'.

In [ ]:
#!/usr/bin/env python3
# NV05 figure — scaling laws & emergence. 2x2: loss-vs-compute frontier, tokens/param,
# emergence sharp (discontinuous metric) vs gradual (smooth metric). Seeded, [illustrative].
import numpy as np
import matplotlib
# matplotlib.use("Agg")  # inline in Colab
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

np.random.seed(7)

INK = "#1a1d24"; DIM = "#6b7280"; BLUE = "#2f6fed"; GREEN = "#2e9e5b"
RED = "#d4462f"; GOLD = "#b8860b"; GRID = "#d8dde6"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 12,
    "axes.edgecolor": "#c2c8d2", "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "axes.titlesize": 13, "axes.titleweight": "bold",
})

fig, ax = plt.subplots(2, 2, figsize=(11.0, 8.0), facecolor="white")
fig.subplots_adjust(hspace=0.42, wspace=0.30, left=0.085, right=0.965, top=0.92, bottom=0.085)

# ---------- Panel 1: loss vs training compute (log-log power-law frontier) ----------
a1 = ax[0, 0]
C = np.logspace(19, 25, 200)                  # FLOPs
# power law: L = k * C^(-0.05); pick k so L=2.6 at C=1e21
k = 2.6 / (1e21 ** -0.05)
L = k * C ** -0.05
a1.plot(C, L, color=BLUE, lw=2.6, zorder=3, label="compute-optimal frontier")
# extrapolation dashed continuation
Cx = np.logspace(24, 25.4, 30); Lx = k * Cx ** -0.05
a1.plot(Cx, Lx, color=BLUE, lw=1.8, ls=(0, (5, 4)), alpha=0.6, zorder=2)
# GPT-3: C ~ 3.15e23, ABOVE the line (under-trained -> higher loss)
c_gpt3 = 3.15e23; l_on = k * c_gpt3 ** -0.05
a1.scatter([c_gpt3], [l_on * 1.085], s=120, color=RED, zorder=5, edgecolor="white", lw=1.2)
a1.annotate("GPT-3\n(~1.7 tok/param,\nunder-trained)", (c_gpt3, l_on * 1.085),
            xytext=(c_gpt3 * 0.10, l_on * 1.16), fontsize=10, color=RED, ha="center",
            arrowprops=dict(arrowstyle="->", color=RED, lw=1.3))
# Chinchilla: C ~ 5.88e23, ON the line
c_chin = 5.88e23; l_chin = k * c_chin ** -0.05
a1.scatter([c_chin], [l_chin], s=120, color=GREEN, zorder=5, edgecolor="white", lw=1.2)
a1.annotate("Chinchilla\n(~20 tok/param,\non frontier)", (c_chin, l_chin),
            xytext=(c_chin * 6.0, l_chin * 1.05), fontsize=10, color=GREEN, ha="center",
            arrowprops=dict(arrowstyle="->", color=GREEN, lw=1.3))
a1.set_xscale("log"); a1.set_yscale("log")
a1.set_xlabel("training compute  C  (FLOPs, log)")
a1.set_ylabel("test loss  (log)")
a1.set_title("Loss vs training compute  —  a straight line")
a1.grid(True, which="both", color=GRID, lw=0.7, alpha=0.7)
a1.text(0.02, 0.04, "[illustrative frontier]", transform=a1.transAxes, fontsize=8.5,
        color=DIM, style="italic")

# ---------- Panel 2: tokens per parameter (rising = over-training) ----------
a2 = ax[0, 1]
names = ["GPT-3", "Chinchilla\noptimal", "Llama 3\n8B", "Qwen3\n0.6B"]
tpp = [1.7, 20, 1875, 60000]
cols = [RED, GREEN, BLUE, GOLD]
xb = np.arange(len(names))
a2.bar(xb, tpp, color=cols, width=0.62, zorder=3)
a2.set_yscale("log")
a2.set_ylim(1, 2e5)
for x, v in zip(xb, tpp):
    lbl = f"{v:,.0f}" if v >= 10 else f"{v:.1f}"
    a2.text(x, v * 1.35, lbl + " : 1", ha="center", fontsize=10, fontweight="bold", color=INK)
a2.axhline(20, color=GREEN, ls="--", lw=1.3, alpha=0.7)
a2.set_xticks(xb); a2.set_xticklabels(names, fontsize=10)
a2.set_ylabel("training tokens per parameter (log)")
a2.set_title("Tokens / parameter  —  rising = over-training")
a2.grid(True, axis="y", which="both", color=GRID, lw=0.7, alpha=0.6)

# ---------- Panel 3: emergence under a discontinuous metric (sharp) ----------
a3 = ax[1, 0]
logN = np.linspace(8, 12, 200)                 # log10 parameters
thr = 10.4
acc = 1.0 / (1.0 + np.exp(-6.0 * (logN - thr)))   # steep sigmoid ~ step
acc = acc * 0.82 + np.random.normal(0, 0.006, size=logN.shape)
acc = np.clip(acc, 0, 0.85)
a3.plot(10 ** logN, acc, color=RED, lw=2.6, zorder=3)
a3.axvline(10 ** thr, color=DIM, ls=":", lw=1.4)
a3.set_xscale("log")
a3.set_xlabel("model scale  (parameters, log)")
a3.set_ylabel("exact-match accuracy")
a3.set_title("Discontinuous metric  —  a sharp 'jump'")
a3.set_ylim(-0.03, 0.92)
a3.grid(True, which="both", color=GRID, lw=0.7, alpha=0.6)
a3.text(0.03, 0.88, "looks emergent", transform=a3.transAxes, fontsize=10, color=RED)

# ---------- Panel 4: same runs under a smooth metric (gradual) ----------
a4 = ax[1, 1]
# per-token log-likelihood improving smoothly (rises gradually across the whole range)
ll = -3.4 + 0.30 * (logN - 8) + np.random.normal(0, 0.012, size=logN.shape)
a4.plot(10 ** logN, ll, color=GREEN, lw=2.6, zorder=3)
a4.axvline(10 ** thr, color=DIM, ls=":", lw=1.4)
a4.set_xscale("log")
a4.set_xlabel("model scale  (parameters, log)")
a4.set_ylabel("per-token log-likelihood")
a4.set_title("Smooth metric  —  gradual all along")
a4.grid(True, which="both", color=GRID, lw=0.7, alpha=0.6)
a4.text(0.03, 0.90, "same models, no jump", transform=a4.transAxes, fontsize=10, color=GREEN)

fig.suptitle("Scaling laws & the emergence 'mirage'   [illustrative — fixed seed]",
             fontsize=14, fontweight="bold", y=0.975)

out = "nv05.png"
fig.savefig(out, dpi=130, facecolor="white")
print("saved", out)

# ---- sanity: the numbers the figure MUST hit ----
def train_flops(params, tokens):
    return 6 * params * tokens
print(f"C(8B,15T) = {train_flops(8e9, 15e12):.2e} FLOPs  (target ~7.2e23)")
print("Llama3-8B tokens/param =", round(15e12 / 8e9))
print("GPT-3 tokens/param     =", round(300e9 / 175e9, 2))
print("Chinchilla tokens/param=", round(1.4e12 / 70e9))
print("tokens/param bars      =", tpp)
plt.show()


## NV10 — Scaled Dot-Product Attention from Scratch

Causal attention in a dozen lines of PyTorch: QKᵀ/√d, mask the future, softmax the rows, weight V — then check it matches the fused F.scaled_dot_product_attention to 1e-3.

In [ ]:
import torch
import torch.nn.functional as F
torch.manual_seed(0)

B, h, T, d = 1, 4, 5, 16                  # batch, heads, seq, head-dim
q, k, v = [torch.randn(B, h, T, d) for _ in range(3)]

def attention(q, k, v):                   # scaled dot-product, causal
    d_k = q.size(-1)
    scores = (q @ k.transpose(-2, -1)) / d_k ** 0.5     # [B,h,T,T]
    mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
    scores = scores.masked_fill(mask, float("-inf"))    # block the future
    w = scores.softmax(dim=-1)                           # rows sum to 1
    return w @ v, w, scores

manual, w, masked = attention(q, k, v)
fused = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("scores grid: ", tuple((q @ k.transpose(-2, -1)).shape), "[B, h, T, T]")
print("row 2 masked:", [round(x, 1) if x > -1e30 else "-inf" for x in masked[0, 0, 2].tolist()])
print("row 2 weights:", [round(x, 2) for x in w[0, 0, 2].tolist()], "sum =", round(w[0, 0, 2].sum().item(), 3))
print("manual out:  ", tuple(manual.shape))
print("fused out:   ", tuple(fused.shape))
print("allclose:    ", torch.allclose(manual, fused, atol=1e-3))


## NV16 — Reading the Spec Sheet — config.json

Every model's config.json is a fingerprint. This reads the real published fields for GPT-2, Llama 3.1, DeepSeek-V3, Qwen3 and Mixtral and classifies each: MHA/GQA/MQA/MLA, dense/MoE, RoPE θ, vocab.

In [ ]:
# NV16 — read config.json fingerprints. Tries live AutoConfig; falls back to the real
# published config.json values (offline) for gated models. Same fingerprint logic either way.
REAL = {  # the actual published config.json fields
  "gpt2":                          dict(num_attention_heads=12, vocab_size=50257),  # learned pos, MHA
  "meta-llama/Llama-3.1-8B":       dict(num_attention_heads=32, num_key_value_heads=8, rope_theta=500000.0, vocab_size=128256),
  "deepseek-ai/DeepSeek-V3":       dict(num_attention_heads=128, kv_lora_rank=512, n_routed_experts=256, num_experts_per_tok=8, n_shared_experts=1, rope_theta=10000.0, vocab_size=129280),
  "Qwen/Qwen3-8B":                 dict(num_attention_heads=32, num_key_value_heads=8, rope_theta=1000000.0, vocab_size=151936),
  "mistralai/Mixtral-8x7B-v0.1":   dict(num_attention_heads=32, num_key_value_heads=8, num_local_experts=8, num_experts_per_tok=2, rope_theta=1000000.0, vocab_size=32000),
}

def get(name):
    try:
        from transformers import AutoConfig
        c = AutoConfig.from_pretrained(name).to_dict()
        return c
    except Exception:
        return REAL[name]  # gated/offline: real published values

def fingerprint(name):
    c = get(name)
    nh  = c["num_attention_heads"]
    kv  = c.get("num_key_value_heads")
    mla = c.get("kv_lora_rank")
    moe = c.get("n_routed_experts") or c.get("num_local_experts")
    attn = ("MLA" if mla else "MHA" if kv in (None, nh) else "MQA" if kv == 1 else f"GQA {nh}:{kv}")
    ffn  = "MoE" if moe else "dense"
    rope = c.get("rope_theta") or "absolute"
    print(f"{name:32} | {attn:9} | {ffn:5} | rope {rope!s:10} | vocab {c['vocab_size']}")

for m in ["gpt2", "meta-llama/Llama-3.1-8B", "deepseek-ai/DeepSeek-V3", "Qwen/Qwen3-8B", "mistralai/Mixtral-8x7B-v0.1"]:
    fingerprint(m)


## NV22 — QLoRA — NF4 + Paged (memory math)

Why a 33B model fits on one 24GB card: 4-bit NF4 weights (~0.5 GB/B vs ~2 GB/B in fp16) plus a tiny LoRA adapter (~0.1% trainable). The math and a representative training trace (GPU-bound, illustrative).

In [ ]:
# nv22_qlora.py — QLoRA setup for NV22 episode [illustrative]
# Shows BitsAndBytesConfig + LoraConfig + SFTConfig fields
# We print representative output (no GPU available)
import torch

# === BitsAndBytesConfig fields ===
print("=== QLoRA Setup (BitsAndBytesConfig) ===")
print(f"load_in_4bit       = True")
print(f"bnb_4bit_quant_type= nf4")
print(f"double_quant       = True")
print(f"compute_dtype      = torch.bfloat16")
print()

# === Memory math ===
params_b = 33
fp16_gb = params_b * 2  # ~2 GB/B
nf4_gb = params_b * 0.5  # ~0.5 GB/B
print(f"model: {params_b}B parameters")
print(f"fp16 load:  ~{fp16_gb:.0f} GB  (does NOT fit 24 GB card)")
print(f"NF4 load:   ~{nf4_gb:.1f} GB  (fits with room for adapter)")
print()

# === Representative terminal output ===
print("loading model-33b in 4-bit (nf4 + double-quant) ....  5.1 GB on GPU")
print("prepare_model_for_kbit_training: gradient checkpointing ON")
print("applying LoRA: all-linear  r=16  alpha=32")
print("trainable params: 33,554,432 || all params: 33,000,000,000 || trainable%: 0.10")
print("optim=paged_adamw_8bit   peak GPU mem: 22.8 GB / 24.0 GB")
print("epoch 3 | step 1500 | loss 0.94 | tok/s 1.7k        [illustrative]")
print("save_pretrained -> out/adapter   (38 MB)")


## NV29 — Chunking & Embeddings

The ingestion half of RAG: a real RecursiveCharacterTextSplitter cuts a document into overlapping chunks, then cosine similarity ranks them against a query (embeddings illustrative — NIM is API/GPU-bound).

In [ ]:
#!/usr/bin/env python3
"""NV29 — Chunking + embeddings demo.
Real RecursiveCharacterTextSplitter (langchain). Cosine sim with seeded synthetic
embeddings (NVIDIA NIM is GPU/API-bound). Marked [illustrative].
"""
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter

# A longer doc to get ~7 chunks at chunk_size=512
document = """Our refund policy allows returns within 30 days of purchase. Items must be in original condition with proof of purchase. Refunds are issued to the original payment method within 5-7 business days after we receive the returned item. If the item is damaged or defective, we will cover return shipping costs. For exchanges, the replacement item will be shipped once we receive the original. Refund requests can be submitted through your account dashboard or by contacting customer support directly.

Our store hours are Monday to Friday, 9am to 6pm, and Saturday 10am to 4pm. We are closed on Sundays and public holidays. During holiday seasons, hours may be extended. Please check our website or social media channels for the latest updates on special holiday hours and any temporary closures due to maintenance or events in the area.

For shipping, we offer standard delivery taking 5-7 business days and express delivery taking 1-2 business days. Free shipping applies to orders over 50 dollars. International shipping is available to select countries with delivery times varying by destination. Tracking information is provided via email once your order ships. Signature confirmation is required for orders exceeding 200 dollars in value.

Our warranty covers manufacturing defects for 12 months from the date of purchase. The warranty does not cover damage from misuse, accidents, or unauthorized modifications. To file a warranty claim, contact our support team with your order number and photos of the defect. Approved claims will receive a replacement or store credit within 10 business days.

Customer support is available via email, phone, and live chat. Email responses are typically within 24 hours. Phone support is available during business hours at our toll-free number. Live chat is available on our website during extended hours including weekends. For urgent issues, priority support is available for premium plan subscribers.

We accept Visa, Mastercard, American Express, and PayPal. Payment is processed securely through our encrypted payment gateway. Gift cards and store credit can also be used at checkout. Installment payment plans are available for orders over 100 dollars through our financing partner. All transactions are protected by industry-standard encryption.

Our loyalty program rewards repeat customers with points on every purchase. Points can be redeemed for discounts on future orders. Premium members earn double points and get early access to sales and new products. Refer a friend and both of you receive bonus points when they make their first purchase. Membership tiers include Silver, Gold, and Platinum with increasing benefits at each level."""

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
chunks = splitter.split_text(document)
print(f"[split] 1 document -> {len(chunks)} chunks  (size<=512, overlap=50, recursive)")
for i, c in enumerate(chunks):
    print(f"  chunk {i}: {len(c)} chars | {c[:48]}...")

# --- EMBED (illustrative) ---
dim = 1024

# Pre-assign cosine similarities we want:
# chunk 0 (refund) -> 0.89
# chunk 3 (warranty/claim) -> 0.41
# chunk 1 (store hours) -> 0.12
# others somewhere between 0.05-0.30
# Build vectors that achieve these cosines

np.random.seed(29)
query_vec = np.random.randn(dim).astype(np.float32)
query_vec /= np.linalg.norm(query_vec)

target_cosines = {}
for i, c in enumerate(chunks):
    cl = c.lower()
    if "refund" in cl and "return" in cl:
        target_cosines[i] = 0.89
    elif "warranty" in cl or "claim" in cl:
        target_cosines[i] = 0.41
    elif "store hours" in cl or "monday" in cl:
        target_cosines[i] = 0.12
    elif "shipping" in cl:
        target_cosines[i] = 0.18
    elif "support" in cl:
        target_cosines[i] = 0.25
    elif "payment" in cl or "visa" in cl:
        target_cosines[i] = 0.09
    elif "loyalty" in cl:
        target_cosines[i] = 0.06
    else:
        target_cosines[i] = 0.15

def make_vec_with_cosine(qv, target_cos, seed):
    """Create a unit vector with approximately target_cos cosine to qv."""
    np.random.seed(seed)
    rand = np.random.randn(dim).astype(np.float32)
    # Orthogonalize rand w.r.t. qv
    rand -= (rand @ qv) * qv
    rand /= np.linalg.norm(rand)
    # v = cos * qv + sqrt(1-cos^2) * rand
    v = target_cos * qv + np.sqrt(1 - target_cos**2) * rand
    v /= np.linalg.norm(v)
    return v

passage_vecs = []
for i in range(len(chunks)):
    tc = target_cosines.get(i, 0.15)
    v = make_vec_with_cosine(query_vec, tc, seed=100+i)
    passage_vecs.append(v)

print(f"[embed] passages: {len(chunks)} x {dim}-dim  via nvidia/llama-3.2-nv-embedqa-1b-v2  (:8080)")

query = "how do I get a refund"
print(f"[embed] query: 1 x {dim}-dim   input_type=query")

def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

ranked = sorted(zip(chunks, passage_vecs), key=lambda cv: -cos(query_vec, cv[1]))
print(f'[rank]  cosine vs query "how do I get a refund":')
for c, v in ranked[:3]:
    print(f"  {cos(query_vec, v):.2f} | {c[:48]}...")

print("[illustrative]")


---
*Built from the real episode scripts. The full video course walks each of these line by line, plus 41 more episodes on architecture, training, RAG, agents, serving and production. — @kader_mohideen*